# 01 Data Loading and Preprocessing


This notebook creates the cleaned data layer used by the rest of the pipeline. It parses the raw workbook, removes non-data interval rows, patches the five missing August Portfolio D daily anchors, and builds the repaired call-volume training layer.

The important design choice is that Stage 1 stays lightweight: August daily `Call Volume`, `CCT`, and `Abandon Rate` are treated as the authoritative daily anchors, with only the five missing Portfolio D `Call Volume` and `CCT` values imputed.

## Core Implementation Used Here

The cells below call `src/pipeline.py` so the notebooks and command-line runner stay consistent. For reviewability, this section shows the exact source code for the functions used in this notebook.

```python
def load_raw_data() -> RawData:
    xlsx = pd.ExcelFile(RAW_DATA_PATH)
    template = pd.read_csv(TEMPLATE_PATH)
    daily: Dict[str, pd.DataFrame] = {}
    intervals: Dict[str, pd.DataFrame] = {}
    mmap = _month_map()

    for portfolio in PORTFOLIOS:
        daily_df = pd.read_excel(xlsx, f"{portfolio} - Daily")
        daily_df.columns = [str(c).strip() for c in daily_df.columns]
        daily_df["Date"] = pd.to_datetime(
            daily_df["Date"].astype(str).str.strip().str.rsplit(" ", n=1).str[0],
            format="%m/%d/%y",
        )
        for col in ["Call Volume", "CCT", "Service Level", "Abandon Rate"]:
            daily_df[col] = pd.to_numeric(daily_df[col], errors="coerce")
        daily_df["Portfolio"] = portfolio
        daily[portfolio] = daily_df.sort_values("Date").reset_index(drop=True)

        interval_df = pd.read_excel(xlsx, f"{portfolio} - Interval")
        interval_df.columns = [str(c).strip() for c in interval_df.columns]
        interval_df = interval_df.dropna(subset=["Interval"]).copy()
        interval_df["mnum"] = interval_df["Month"].map(mmap)
        interval_df["Day"] = pd.to_numeric(interval_df["Day"], errors="coerce").astype(int)
        interval_df["Date"] = pd.to_datetime(dict(year=2025, month=interval_df["mnum"], day=interval_df["Day"]))
        interval_df["slot"] = interval_df["Interval"].apply(lambda t: int(t.hour * 2 + t.minute // 30))
        for col in ["Call Volume", "Abandoned Calls", "Abandoned Rate", "CCT", "Service Level"]:
            interval_df[col] = pd.to_numeric(interval_df[col], errors="coerce")
        interval_df["Portfolio"] = portfolio
        intervals[portfolio] = interval_df.sort_values(["Date", "slot"]).reset_index(drop=True)

    return RawData(daily=daily, intervals=intervals, template=template)

def stack_daily(daily: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    cols = ["Portfolio", "Date", "Call Volume", "CCT", "Service Level", "Abandon Rate"]
    return pd.concat([df[cols] for df in daily.values()], ignore_index=True).sort_values(["Portfolio", "Date"])

def stack_intervals(intervals: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    cols = [
        "Portfolio",
        "Date",
        "slot",
        "Interval",
        "Call Volume",
        "Abandoned Calls",
        "Abandoned Rate",
        "CCT",
        "Service Level",
    ]
    return pd.concat([df[cols] for df in intervals.values()], ignore_index=True).sort_values(["Portfolio", "Date", "slot"])

def patch_august_daily_anchors(daily: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    august = pd.date_range(AUGUST_START, AUGUST_END, freq="D")
    anchors = []
    for portfolio in PORTFOLIOS:
        p_aug = (
            daily[portfolio]
            .loc[(daily[portfolio]["Date"] >= AUGUST_START) & (daily[portfolio]["Date"] <= AUGUST_END)]
            .set_index("Date")
            .reindex(august)
            .copy()
        )
        p_aug["Portfolio"] = portfolio
        anchors.append(
            p_aug[["Portfolio", "Call Volume", "CCT", "Abandon Rate"]]
            .rename_axis("Date")
            .reset_index()
        )
    anchors_df = pd.concat(anchors, ignore_index=True)

    daily_map = {p: daily[p].set_index("Date") for p in PORTFOLIOS}
    missing_mask = (
        (anchors_df["Portfolio"] == "D")
        & (anchors_df["Date"] >= pd.Timestamp("2025-08-27"))
        & (anchors_df["Date"] <= pd.Timestamp("2025-08-31"))
    )

    for idx, row in anchors_df.loc[missing_mask].iterrows():
        dt = row["Date"]
        prev = dt - pd.Timedelta(days=7)
        abc_cv_now = sum(float(daily_map[p].loc[dt, "Call Volume"]) for p in ["A", "B", "C"])
        abc_cv_prev = sum(float(daily_map[p].loc[prev, "Call Volume"]) for p in ["A", "B", "C"])
        d_cv_prev = float(daily_map["D"].loc[prev, "Call Volume"])
        anchors_df.loc[idx, "Call Volume"] = round(d_cv_prev * abc_cv_now / abc_cv_prev)

        abc_cct_now = np.mean([float(daily_map[p].loc[dt, "CCT"]) for p in ["A", "B", "C"]])
        abc_cct_prev = np.mean([float(daily_map[p].loc[prev, "CCT"]) for p in ["A", "B", "C"]])
        d_cct_prev = float(daily_map["D"].loc[prev, "CCT"])
        adjusted = abc_cct_now + (d_cct_prev - abc_cct_prev)
        anchors_df.loc[idx, "CCT"] = round(0.7 * d_cct_prev + 0.3 * adjusted, 2)

    anchors_df["daily_abandoned_calls"] = anchors_df["Call Volume"] * anchors_df["Abandon Rate"]
    return anchors_df.sort_values(["Portfolio", "Date"]).reset_index(drop=True)

def create_full_interval_grid(portfolio: str, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    dates = pd.date_range(start, end, freq="D")
    grid = pd.MultiIndex.from_product([dates, range(SLOTS_PER_DAY)], names=["Date", "slot"]).to_frame(index=False)
    grid["Portfolio"] = portfolio
    grid["dow"] = grid["Date"].dt.dayofweek
    grid["dom"] = grid["Date"].dt.day
    grid["is_peak"] = ((grid["slot"] // 2 >= 9) & (grid["slot"] // 2 <= 17)).astype(int)
    grid["slot_sin"] = np.sin(2 * np.pi * grid["slot"] / SLOTS_PER_DAY)
    grid["slot_cos"] = np.cos(2 * np.pi * grid["slot"] / SLOTS_PER_DAY)
    grid["dow_sin"] = np.sin(2 * np.pi * grid["dow"] / 7)
    grid["dow_cos"] = np.cos(2 * np.pi * grid["dow"] / 7)
    grid["dom_sin"] = np.sin(2 * np.pi * grid["dom"] / 31)
    grid["dom_cos"] = np.cos(2 * np.pi * grid["dom"] / 31)
    return grid

def build_cv_share_prior(observed_grid: pd.DataFrame) -> tuple[Dict[tuple[str, int], np.ndarray], Dict[str, np.ndarray], np.ndarray]:
    train = observed_grid.dropna(subset=["daily_cv", "Call Volume"]).copy()
    train = train[(train["daily_cv"] > 0) & (train["Call Volume"] >= 0)]
    train["observed_share"] = train["Call Volume"] / train["daily_cv"]

    portfolio_slot = {}
    for portfolio in PORTFOLIOS:
        p = train[train["Portfolio"] == portfolio]
        med = p.groupby("slot")["observed_share"].median()
        arr = np.array([med.get(slot, np.nan) for slot in range(SLOTS_PER_DAY)], dtype=float)
        portfolio_slot[portfolio] = smooth_share(np.nan_to_num(arr, nan=0.0))

    global_med = train.groupby("slot")["observed_share"].median()
    global_profile = smooth_share([global_med.get(slot, 0.0) for slot in range(SLOTS_PER_DAY)])

    priors: Dict[tuple[str, int], np.ndarray] = {}
    for portfolio in PORTFOLIOS:
        for dow in range(7):
            sub = train[(train["Portfolio"] == portfolio) & (train["dow"] == dow)]
            med = sub.groupby("slot")["observed_share"].median()
            arr = np.array([med.get(slot, np.nan) for slot in range(SLOTS_PER_DAY)], dtype=float)
            fallback = portfolio_slot.get(portfolio, global_profile)
            arr = np.where(np.isfinite(arr), arr, fallback)
            priors[(portfolio, dow)] = smooth_share(arr)
    return priors, portfolio_slot, global_profile

def build_cv_training_layer(daily_all: pd.DataFrame, interval_all: pd.DataFrame) -> pd.DataFrame:
    rows = []
    train_daily = daily_all[(daily_all["Date"] >= TRAIN_START) & (daily_all["Date"] <= TRAIN_END)].copy()
    train_intervals = interval_all[(interval_all["Date"] >= TRAIN_START) & (interval_all["Date"] <= TRAIN_END)].copy()

    observed_grids = []
    for portfolio in PORTFOLIOS:
        grid = create_full_interval_grid(portfolio, TRAIN_START, TRAIN_END)
        observed = train_intervals[train_intervals["Portfolio"] == portfolio][
            ["Date", "slot", "Call Volume"]
        ].copy()
        p_daily = train_daily[train_daily["Portfolio"] == portfolio][["Date", "Call Volume"]].rename(columns={"Call Volume": "daily_cv"})
        grid = grid.merge(observed, on=["Date", "slot"], how="left")
        grid = grid.merge(p_daily, on="Date", how="left")
        observed_grids.append(grid)
    observed_grid = pd.concat(observed_grids, ignore_index=True)
    priors, _portfolio_slot, _global_profile = build_cv_share_prior(observed_grid)

    for (portfolio, dt), day in observed_grid.groupby(["Portfolio", "Date"], sort=True):
        day = day.sort_values("slot").reset_index(drop=True).copy()
        daily_cv = day["daily_cv"].iloc[0]
        prior = priors[(portfolio, int(day["dow"].iloc[0]))]
        valid = day["Call Volume"].notna() & (day["Call Volume"] >= 0)
        observed_sum = float(day.loc[valid, "Call Volume"].sum())
        observed_share_mass = float(prior[valid.to_numpy()].sum())
        observed_count = int(valid.sum())

        if pd.notna(daily_cv) and daily_cv > 0:
            total = float(daily_cv)
            repaired = np.zeros(SLOTS_PER_DAY, dtype=float)
            if observed_sum > total or observed_count == SLOTS_PER_DAY:
                repaired = np.clip(day["Call Volume"].fillna(0).to_numpy(dtype=float), 0, None)
                repaired = repaired * (total / repaired.sum()) if repaired.sum() > 0 else total * prior
                case = "A_scale_to_daily_anchor"
            else:
                repaired[valid.to_numpy()] = day.loc[valid, "Call Volume"].to_numpy(dtype=float)
                missing = ~valid.to_numpy()
                remainder = max(total - repaired.sum(), 0.0)
                missing_mass = float(prior[missing].sum())
                if missing.any() and missing_mass > 0:
                    repaired[missing] = remainder * prior[missing] / missing_mass
                case = "A_daily_anchor"
        elif observed_count >= 42 and observed_share_mass >= 0.80 and observed_sum > 0:
            total = observed_sum / observed_share_mass
            repaired = np.zeros(SLOTS_PER_DAY, dtype=float)
            repaired[valid.to_numpy()] = day.loc[valid, "Call Volume"].to_numpy(dtype=float)
            missing = ~valid.to_numpy()
            repaired[missing] = total * prior[missing]
            case = "B_estimated_daily_total"
        else:
            continue

        day["repaired_cv"] = repaired
        day["daily_cv"] = float(repaired.sum())
        day["slot_share"] = day["repaired_cv"] / day["daily_cv"]
        day["repair_case"] = case
        day["observed_slot_count"] = observed_count
        rows.append(day)

    return pd.concat(rows, ignore_index=True).sort_values(["Portfolio", "Date", "slot"])

def save_preprocessed_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    ensure_dirs()
    raw = load_raw_data()
    daily_all = stack_daily(raw.daily)
    interval_all = stack_intervals(raw.intervals)
    anchors = patch_august_daily_anchors(raw.daily)
    cv_training = build_cv_training_layer(daily_all, interval_all)

    daily_all.to_csv(PROCESSED_DIR / "daily_clean.csv", index=False)
    interval_all.to_csv(PROCESSED_DIR / "interval_clean.csv", index=False)
    anchors.to_csv(PROCESSED_DIR / "august_daily_anchors.csv", index=False)
    cv_training.to_csv(PROCESSED_DIR / "cv_training_layer.csv", index=False)
    raw.template.to_csv(PROCESSED_DIR / "template_columns.csv", index=False)
    return daily_all, interval_all, anchors, cv_training
```


In [ ]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'src' / 'pipeline.py').exists():
        ROOT = candidate
        break
    if (candidate / 'datathon_final' / 'src' / 'pipeline.py').exists():
        ROOT = candidate / 'datathon_final'
        break
if ROOT is None:
    raise RuntimeError('Could not find datathon_final project root.')
sys.path.insert(0, str(ROOT))
from src import pipeline


## Generate processed data

Outputs written to `data/processed/`:

- `daily_clean.csv`: parsed daily sheets for portfolios A-D
- `interval_clean.csv`: parsed Apr-Jun interval records with true 2025 dates and slot indices
- `august_daily_anchors.csv`: completed August daily anchors, including the five Portfolio D patches
- `cv_training_layer.csv`: repaired 48-slot call-volume training layer

In [ ]:
daily_all, interval_all, anchors, cv_training = pipeline.save_preprocessed_data()
print(f'Daily rows: {len(daily_all):,}')
print(f'Interval rows: {len(interval_all):,}')
print(f'August anchor rows: {len(anchors):,}')
print(f'CV training rows: {len(cv_training):,}')

## Stage 1 anchor patch

Portfolio D was missing daily `Call Volume` and `CCT` from August 27-31, 2025. `Call Volume` was patched from Portfolio D's previous-week share of the observed A+B+C total. `CCT` was patched from Portfolio D's previous-week value, lightly adjusted by the A+B+C week-over-week movement. Daily `Abandon Rate` was already present and was kept as-is.

In [ ]:
anchors.loc[(anchors['Portfolio'] == 'D') & (anchors['Date'] >= '2025-08-27'),
            ['Portfolio', 'Date', 'Call Volume', 'CCT', 'Abandon Rate', 'daily_abandoned_calls']]

## CV training repair cases

The call-volume training layer is repaired day by day. When daily `CV` exists, it controls the day total and missing slots are filled from historical slot-share priors. When daily `CV` is missing but the interval day is still repairable, the daily total is estimated from observed historical share mass. Days that are too synthetic are dropped.

In [ ]:
cv_training.groupby(['Portfolio', 'repair_case']).size().rename('rows').reset_index()